<a href="https://colab.research.google.com/github/viraj97-sl/ai-ml-ds-learning-hub/blob/master/02_ML_Engineer/beginner/04_fastapi_intro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FastAPI for ML Serving

FastAPI is the modern standard for serving ML models via REST APIs. It provides automatic documentation, Pydantic validation, async support, and high performance.

**Topics:** FastAPI basics, Pydantic validation, async endpoints, background tasks, middleware, testing

In [ ]:
# Install: pip install fastapi uvicorn[standard] pydantic
try:
    import fastapi
    print(f'FastAPI {fastapi.__version__} available')
except ImportError:
    print('Install: pip install fastapi uvicorn[standard]')

# We'll write the full application code as strings for documentation
# Save to src/main.py and run with: uvicorn src.main:app --reload

## 1. Complete ML API Application

In [ ]:
fastapi_app = '''
# src/main.py — Production ML API
from fastapi import FastAPI, HTTPException, BackgroundTasks, Depends
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse
from pydantic import BaseModel, Field, validator
from typing import List, Optional, Dict, Any
import time
import logging
import joblib
import numpy as np

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Initialize FastAPI app with metadata
app = FastAPI(
    title="ML Inference API",
    description="Credit risk scoring API powered by GBM model",
    version="2.1.0",
    docs_url="/docs",      # Swagger UI
    redoc_url="/redoc",    # ReDoc UI
)

# CORS middleware (allow browser requests)
app.add_middleware(
    CORSMiddleware,
    allow_origins=["https://app.company.com"],
    allow_methods=["POST", "GET"],
    allow_headers=["*"],
)

# ===== Pydantic Models =====

class CreditApplication(BaseModel):
    """Credit scoring request with full validation."""
    customer_id: str = Field(..., description="Unique customer identifier")
    age: int = Field(..., ge=18, le=100, description="Applicant age")
    annual_income: float = Field(..., gt=0, description="Annual income in USD")
    credit_score: int = Field(..., ge=300, le=850, description="FICO credit score")
    employment_years: float = Field(..., ge=0, description="Years at current employer")
    existing_debt: float = Field(default=0.0, ge=0, description="Total existing debt")
    
    @validator('annual_income')
    def income_reasonable(cls, v):
        if v > 10_000_000:
            raise ValueError('Income exceeds maximum allowed value')
        return v

class PredictionResponse(BaseModel):
    """Credit scoring response."""
    customer_id: str
    risk_score: float = Field(..., ge=0, le=1, description="Default probability")
    risk_tier: str = Field(..., description="LOW / MEDIUM / HIGH")
    recommended_limit: float = Field(..., ge=0, description="Credit limit recommendation")
    model_version: str
    latency_ms: float

class BatchRequest(BaseModel):
    applications: List[CreditApplication]
    priority: Optional[str] = "normal"

class HealthResponse(BaseModel):
    status: str
    model_version: str
    uptime_seconds: float
    predictions_served: int

# ===== State =====
class ModelRegistry:
    def __init__(self):
        self.model = None
        self.version = None
        self.predictions_served = 0
        self.startup_time = time.time()

registry = ModelRegistry()

# ===== Startup / Shutdown =====
@app.on_event("startup")
async def load_model():
    """Load model at startup — not on each request!"""
    logger.info("Loading ML model...")
    # registry.model = joblib.load("models/credit_model_v2.pkl")
    registry.model = {"weights": "loaded"}  # mock for demo
    registry.version = "2.1.0"
    logger.info(f"Model v{registry.version} loaded")

@app.on_event("shutdown")
async def cleanup():
    logger.info("Shutting down — cleaning up resources")

# ===== Endpoints =====
@app.get("/health", response_model=HealthResponse)
async def health_check():
    """Health check endpoint for load balancer / k8s probes."""
    return HealthResponse(
        status="healthy",
        model_version=registry.version,
        uptime_seconds=time.time() - registry.startup_time,
        predictions_served=registry.predictions_served,
    )

@app.post("/predict", response_model=PredictionResponse)
async def predict(application: CreditApplication):
    """Single application credit scoring."""
    if registry.model is None:
        raise HTTPException(status_code=503, detail="Model not loaded")
    
    start = time.perf_counter()
    try:
        # Extract features
        features = np.array([[
            application.age,
            application.annual_income,
            application.credit_score,
            application.employment_years,
            application.existing_debt / max(application.annual_income, 1),  # debt ratio
        ]])
        
        # Inference (replace with actual model.predict_proba(features)[0,1])
        risk_score = float(np.clip(0.1 + 0.5 * (850 - application.credit_score) / 550, 0, 1))
        
        # Business logic
        if risk_score < 0.2:
            risk_tier = "LOW"
            credit_limit = min(application.annual_income * 0.4, 50000)
        elif risk_score < 0.5:
            risk_tier = "MEDIUM"
            credit_limit = min(application.annual_income * 0.2, 15000)
        else:
            risk_tier = "HIGH"
            credit_limit = 0.0
        
        registry.predictions_served += 1
        latency_ms = (time.perf_counter() - start) * 1000
        
        logger.info(f"Prediction: customer={application.customer_id} "
                    f"score={risk_score:.3f} tier={risk_tier} lat={latency_ms:.1f}ms")
        
        return PredictionResponse(
            customer_id=application.customer_id,
            risk_score=round(risk_score, 4),
            risk_tier=risk_tier,
            recommended_limit=round(credit_limit, 2),
            model_version=registry.version,
            latency_ms=round(latency_ms, 2),
        )
    except Exception as e:
        logger.error(f"Prediction failed: {e}")
        raise HTTPException(status_code=500, detail="Internal prediction error")

@app.post("/predict/batch")
async def predict_batch(request: BatchRequest, background_tasks: BackgroundTasks):
    """Batch scoring — returns job_id for async processing."""
    job_id = f"batch_{int(time.time())}"
    background_tasks.add_task(process_batch, job_id, request.applications)
    return {"job_id": job_id, "status": "queued", "n_applications": len(request.applications)}

async def process_batch(job_id: str, applications: List[CreditApplication]):
    """Background task: process batch asynchronously."""
    logger.info(f"Processing batch {job_id} ({len(applications)} applications)")
    # ... process and write results to database ...
    logger.info(f"Batch {job_id} complete")
'''

print(fastapi_app)

## 2. Testing FastAPI Endpoints

In [ ]:
test_code = '''
# tests/test_api.py
import pytest
from fastapi.testclient import TestClient
from src.main import app

client = TestClient(app)

class TestHealthEndpoint:
    def test_health_returns_200(self):
        response = client.get("/health")
        assert response.status_code == 200
    
    def test_health_response_schema(self):
        response = client.get("/health")
        data = response.json()
        assert "status" in data
        assert data["status"] == "healthy"
        assert "model_version" in data

class TestPredictEndpoint:
    valid_payload = {
        "customer_id": "CUST-001",
        "age": 35,
        "annual_income": 75000,
        "credit_score": 720,
        "employment_years": 5.0,
        "existing_debt": 15000,
    }
    
    def test_valid_prediction(self):
        response = client.post("/predict", json=self.valid_payload)
        assert response.status_code == 200
        data = response.json()
        assert 0 <= data["risk_score"] <= 1
        assert data["risk_tier"] in ["LOW", "MEDIUM", "HIGH"]
        assert data["customer_id"] == "CUST-001"
    
    def test_invalid_age_rejected(self):
        payload = {**self.valid_payload, "age": 15}  # under 18
        response = client.post("/predict", json=payload)
        assert response.status_code == 422  # Pydantic validation error
    
    def test_invalid_credit_score_rejected(self):
        payload = {**self.valid_payload, "credit_score": 200}  # below 300
        response = client.post("/predict", json=payload)
        assert response.status_code == 422
    
    def test_missing_required_field(self):
        payload = {k: v for k, v in self.valid_payload.items() if k != "customer_id"}
        response = client.post("/predict", json=payload)
        assert response.status_code == 422
    
    def test_high_risk_customer_gets_zero_limit(self):
        payload = {**self.valid_payload, "credit_score": 300}  # terrible credit
        response = client.post("/predict", json=payload)
        assert response.status_code == 200
        assert response.json()["risk_tier"] == "HIGH"
        assert response.json()["recommended_limit"] == 0.0

# Run: pytest tests/test_api.py -v
'''

print('=== API Test Suite ===')
print(test_code)

print('=== Running the API ===')
print('''
Development:
  uvicorn src.main:app --reload --port 8000

Production (multiple workers):
  uvicorn src.main:app --host 0.0.0.0 --port 8000 --workers 4

Or via Gunicorn (more robust process management):
  gunicorn src.main:app -w 4 -k uvicorn.workers.UvicornWorker

API docs auto-generated at:
  http://localhost:8000/docs  → Swagger UI (interactive)
  http://localhost:8000/redoc → ReDoc (clean)
''')